In [47]:
import pandas as pd, glob, os
import numpy as np
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, average_precision_score
import shap

In [36]:
dfs = {}
for f in glob.glob("../data/*.csv"):
    name = os.path.basename(f).replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    dfs[name] = pd.read_csv(f)

print(list(dfs.keys()))
globals().update(dfs)

['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 'sellers', 'product_category_name_translation']


In [37]:
geo = (geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .mean().reset_index())

In [38]:
items = order_items.groupby("order_id").agg(
    freight=("freight_value", "sum"), price=("price", "sum"),
    n_items=("order_item_id", "count"),
    seller_id=("seller_id", "first"), product_id=("product_id", "first")).reset_index()

In [39]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
"order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

In [40]:
delivered = orders[orders.order_status == "delivered"].dropna(
    subset=["order_delivered_customer_date"]).copy()
delivered["is_late"] = (delivered.order_delivered_customer_date >
    delivered.order_estimated_delivery_date).astype(int)
delivered["delay_days"] = (delivered.order_delivered_customer_date -
    delivered.order_estimated_delivery_date).dt.days

In [41]:
df = (delivered.merge(items, on="order_id")
.merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]], on="customer_id")
.merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id")
.merge(products[["product_id", "product_weight_g", "product_length_cm","product_height_cm", "product_width_cm"]], on="product_id"))


In [42]:
def add_coords(d, zip_col, tag):
    g = geo.rename(columns={"geolocation_zip_code_prefix": zip_col,
                            "geolocation_lat": tag + "_lat", 
                            "geolocation_lng": tag + "_lng"})
    return d.merge(g, on=zip_col, how="left")

In [43]:
df = add_coords(df, "customer_zip_code_prefix", "cust")
df = add_coords(df, "seller_zip_code_prefix", "sell")


In [44]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    a = np.sin((lat2-lat1)/2)**2 + \
        np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

In [45]:
df["distance_km"] = haversine(df.cust_lat, df.cust_lng, df.sell_lat, df.sell_lng)


In [46]:
df["hour"]  = df.order_purchase_timestamp.dt.hour
df["dow"]   = df.order_purchase_timestamp.dt.dayofweek
df["month"] = df.order_purchase_timestamp.dt.month
df["same_state"] = (df.customer_state == df.seller_state).astype(int)
df["volume_cm3"] = df.product_length_cm * df.product_height_cm * df.product_width_cm

In [48]:
feats = ["hour", "dow", "month", "freight", "price", "n_items",
        "product_weight_g", "volume_cm3", "distance_km", "same_state"]

In [ ]:
df = df.sort_values("order_purchase_timestamp")
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
Xtr, ytr, Xte, yte = train[feats], train.is_late, test[feats], test.is_late